In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
import h5py
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== Hyperparameters ====================
HYPERPARAMS = {
    'batch_size': 64,
    'num_epochs': 50,
    'initial_lr': 1e-3,
    'weight_decay': 1e-4,
    'patience': 7,
    'min_lr': 1e-6,
    'factor': 0.5,
    'num_workers': 4,
    'grad_clip': 1.0,
    'label_smoothing': 0.1,
    'mixup_alpha': 0.2
}

# ==================== Dataset Class ====================
class PatchCamelyonDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None, augment=False):
        self.x_file = h5py.File(x_path, 'r')
        self.y_file = h5py.File(y_path, 'r')
        self.x_data = self.x_file['x']
        self.y_data = self.y_file['y']
        self.transform = transform
        self.augment = augment
        self.length = len(self.y_data)
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        image = self.x_data[idx]
        label = self.y_data[idx][0][0][0]  # Extract scalar label
        
        # Convert to PIL-compatible format
        image = image.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def __del__(self):
        self.x_file.close()
        self.y_file.close()

# ==================== Attention Module ====================
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = self.sigmoid(avg_out + max_out)
        return x * out

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(out))
        return x * out

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

# ==================== EfficientNet-B3 with Attention ====================
class EfficientNetB3WithAttention(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(EfficientNetB3WithAttention, self).__init__()
        
        # Load pretrained EfficientNet-B3
        if pretrained:
            self.efficientnet = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        else:
            self.efficientnet = efficientnet_b3(weights=None)
        
        # Get feature dimension
        in_features = self.efficientnet.classifier[1].in_features
        
        # Remove original classifier
        self.efficientnet.classifier = nn.Identity()
        
        # Add attention module
        self.attention = CBAM(in_features, reduction=16)
        
        # Custom classifier with dropout
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # Extract features
        x = self.efficientnet.features(x)
        
        # Apply attention
        x = self.attention(x)
        
        # Classify
        x = self.classifier(x)
        
        return x
    
    def get_feature_maps(self, x):
        """For Grad-CAM visualization"""
        return self.efficientnet.features(x)

# ==================== Data Augmentation ====================
def get_transforms(augment=False):
    if augment:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

# ==================== Mixup Augmentation ====================
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ==================== Training Function ====================
def train_epoch(model, train_loader, criterion, optimizer, device, use_mixup=True):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.float().to(device)
        
        # Apply mixup
        if use_mixup and np.random.rand() > 0.5:
            images, labels_a, labels_b, lam = mixup_data(images, labels, HYPERPARAMS['mixup_alpha'])
            
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), HYPERPARAMS['grad_clip'])
        optimizer.step()
        
        running_loss += loss.item()
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# ==================== Validation Function ====================
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).float()
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
            pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    # Calculate ROC-AUC
    roc_auc = roc_auc_score(all_labels, all_probs)
    
    return epoch_loss, epoch_acc, roc_auc, all_preds, all_labels, all_probs

# ==================== Grad-CAM Visualization ====================
def visualize_gradcam(model, image, label, pred_prob, device, save_path='gradcam_output.png'):
    model.eval()
    
    # Prepare target layer
    target_layers = [model.efficientnet.features[-1]]
    
    # Create Grad-CAM object
    cam = GradCAM(model=model, target_layers=target_layers)
    
    # Generate CAM
    input_tensor = image.unsqueeze(0).to(device)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)
    grayscale_cam = grayscale_cam[0, :]
    
    # Denormalize image for visualization
    img_np = image.cpu().numpy().transpose(1, 2, 0)
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    # Create visualization
    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title(f'Original\nTrue Label: {label}')
    axes[0].axis('off')
    
    axes[1].imshow(grayscale_cam, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap')
    axes[1].axis('off')
    
    axes[2].imshow(visualization)
    axes[2].set_title(f'Overlay\nPred Prob: {pred_prob:.3f}')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

# ==================== Main Training Loop ====================
def main():
    # Dataset paths
    train_x_path = 'pcam_dataset/camelyonpatch_level_2_split_train_x.h5'
    train_y_path = 'pcam_dataset/camelyonpatch_level_2_split_train_y.h5'
    valid_x_path = 'pcam_dataset/camelyonpatch_level_2_split_valid_x.h5'
    valid_y_path = 'pcam_dataset/camelyonpatch_level_2_split_valid_y.h5'
    test_x_path = 'pcam_dataset/camelyonpatch_level_2_split_test_x.h5'
    test_y_path = 'pcam_dataset/camelyonpatch_level_2_split_test_y.h5'
    
    # Create datasets
    print("Loading datasets...")
    train_dataset = PatchCamelyonDataset(train_x_path, train_y_path, 
                                         transform=get_transforms(augment=True))
    valid_dataset = PatchCamelyonDataset(valid_x_path, valid_y_path, 
                                         transform=get_transforms(augment=False))
    test_dataset = PatchCamelyonDataset(test_x_path, test_y_path, 
                                        transform=get_transforms(augment=False))
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=HYPERPARAMS['batch_size'], 
                              shuffle=True, num_workers=HYPERPARAMS['num_workers'], 
                              pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=HYPERPARAMS['batch_size'], 
                              shuffle=False, num_workers=HYPERPARAMS['num_workers'], 
                              pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=HYPERPARAMS['batch_size'], 
                             shuffle=False, num_workers=HYPERPARAMS['num_workers'], 
                             pin_memory=True)
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Valid samples: {len(valid_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
    
    # Initialize model
    print("\nInitializing model...")
    model = EfficientNetB3WithAttention(num_classes=1, pretrained=True).to(device)
    
    # Loss function with label smoothing
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0]).to(device))
    
    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=HYPERPARAMS['initial_lr'], 
                            weight_decay=HYPERPARAMS['weight_decay'])
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=HYPERPARAMS['factor'], 
        patience=HYPERPARAMS['patience'], min_lr=HYPERPARAMS['min_lr']
    )
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'valid_loss': [], 'valid_acc': [], 'valid_roc': []
    }
    
    best_roc = 0.0
    patience_counter = 0
    
    print("\n" + "="*50)
    print("Starting Training")
    print("="*50)
    
    # Training loop
    for epoch in range(HYPERPARAMS['num_epochs']):
        print(f"\nEpoch {epoch+1}/{HYPERPARAMS['num_epochs']}")
        print("-" * 50)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        valid_loss, valid_acc, valid_roc, _, _, _ = validate(model, valid_loader, criterion, device)
        
        # Update scheduler
        scheduler.step(valid_roc)
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)
        history['valid_roc'].append(valid_roc)
        
        # Print metrics
        print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.2f}% | Valid ROC-AUC: {valid_roc:.4f}")
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if valid_roc > best_roc:
            best_roc = valid_roc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"✓ Best model saved with ROC-AUC: {best_roc:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= HYPERPARAMS['patience'] * 2:
            print("\nEarly stopping triggered!")
            break
    
    # Load best model for testing
    print("\n" + "="*50)
    print("Testing on Best Model")
    print("="*50)
    model.load_state_dict(torch.load('best_model.pth'))
    
    # Test evaluation
    test_loss, test_acc, test_roc, test_preds, test_labels, test_probs = validate(
        model, test_loader, criterion, device
    )
    
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Test ROC-AUC: {test_roc:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # Plot training history
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['valid_loss'], label='Valid Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
    axes[1].plot(history['valid_acc'], label='Valid Acc', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # ROC-AUC plot
    axes[2].plot(history['valid_roc'], label='Valid ROC-AUC', marker='d', color='green')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('ROC-AUC')
    axes[2].set_title('Validation ROC-AUC Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # Generate Grad-CAM visualizations
    print("\nGenerating Grad-CAM visualizations...")
    model.eval()
    
    # Get some test samples
    test_images, test_targets = next(iter(test_loader))
    
    for i in range(min(5, len(test_images))):
        img = test_images[i]
        label = test_targets[i].item()
        
        with torch.no_grad():
            output = model(img.unsqueeze(0).to(device))
            prob = torch.sigmoid(output).item()
        
        visualize_gradcam(model, img, label, prob, device, 
                         save_path=f'gradcam_sample_{i+1}.png')
    
    print("\n" + "="*50)
    print("Training Complete!")
    print("="*50)
    print(f"Best Validation ROC-AUC: {best_roc:.4f}")
    print(f"Test ROC-AUC: {test_roc:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")

if __name__ == "__main__":
    main()

Using device: cuda
Loading datasets...
Train samples: 262144
Valid samples: 32768
Test samples: 32768

Initializing model...

Starting Training

Epoch 1/50
--------------------------------------------------


Validation: 100%|██████████| 512/512 [00:13<00:00, 39.17it/s, loss=0.404, acc=90.2] 



Train Loss: 0.2986 | Train Acc: 80.53%
Valid Loss: 0.2526 | Valid Acc: 90.16% | Valid ROC-AUC: 0.9615
Current LR: 0.001000
✓ Best model saved with ROC-AUC: 0.9615

Epoch 2/50
--------------------------------------------------


Validation: 100%|██████████| 512/512 [00:13<00:00, 39.14it/s, loss=0.359, acc=90.9] 



Train Loss: 0.2352 | Train Acc: 82.75%
Valid Loss: 0.2446 | Valid Acc: 90.93% | Valid ROC-AUC: 0.9636
Current LR: 0.001000
✓ Best model saved with ROC-AUC: 0.9636

Epoch 3/50
--------------------------------------------------


...


Train Loss: 0.1261 | Train Acc: 86.58%
Valid Loss: 0.2755 | Valid Acc: 90.32% | Valid ROC-AUC: 0.9614
Current LR: 0.000500

Epoch 20/50
--------------------------------------------------


Validation: 100%|██████████| 512/512 [00:06<00:00, 84.46it/s, loss=0.529, acc=88.6]  



Train Loss: 0.1246 | Train Acc: 86.10%
Valid Loss: 0.3042 | Valid Acc: 88.62% | Valid ROC-AUC: 0.9524
Current LR: 0.000500

Early stopping triggered!

Testing on Best Model


Validation: 100%|██████████| 512/512 [00:04<00:00, 103.81it/s, loss=0.444, acc=85.1] 



Test Loss: 0.3369
Test Accuracy: 85.12%
Test ROC-AUC: 0.9579

Generating Grad-CAM visualizations...

Training Complete!
Best Validation ROC-AUC: 0.9692
Test ROC-AUC: 0.9579
Test Accuracy: 85.12%
